In [1]:
from functools import partial
import torch
import numpy as np  
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd

import timm
from transformers import AutoTokenizer
import albumentations as A
from albumentations.pytorch import ToTensorV2  



class MultimodalDataset(Dataset):
    def __init__(self, df, text_model, image_model, transforms):
        self.df = df
        self.image_cfg = timm.get_pretrained_cfg(image_model)
        self.tokenizer = AutoTokenizer.from_pretrained(text_model)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "text"]
        label = self.df.loc[idx, "label"]

        img_path = self.df.loc[idx, "image_path"]
        image = Image.open(f"data/images/{img_path}").convert('RGB')
        image = self.transforms(image=np.array(image))["image"]

        return {"label": label, "image": image, "text": text}


def collate_fn(batch, tokenizer):
    texts = [item["text"] for item in batch]
    images = torch.stack([item["image"] for item in batch])
    labels = torch.LongTensor([item["label"] for item in batch])

    tokenized_input = tokenizer(texts,
                                return_tensors="pt",
                                padding="max_length",
                                truncation=True)
    return {
        "label": labels,
        "image": images,
        "input_ids": tokenized_input["input_ids"],
        "attention_mask": tokenized_input["attention_mask"]
    }


text_model = "bert-base-uncased"
image_model = 'tf_efficientnet_b0'
tokenizer = AutoTokenizer.from_pretrained(text_model)
cfg = timm.get_pretrained_cfg(image_model)

transforms = A.Compose(
    [
        A.SmallestMaxSize(max_size=max(cfg.input_size[1], cfg.input_size[2]),
                          p=1.0),
        A.RandomCrop(height=cfg.input_size[1], width=cfg.input_size[2], p=1.0),
        A.Affine(scale=(0.8, 1.2),
                 rotate=(-15, 15),
                 translate_percent=(-0.1, 0.1),
                 shear=(-10, 10),
                 fill=0,
                 p=0.8),
        A.CoarseDropout(num_holes_range=(2, 8),
                        hole_height_range=(int(0.07 * cfg.input_size[1]),
                                           int(0.15 * cfg.input_size[1])),
                        hole_width_range=(int(0.1 * cfg.input_size[2]),
                                          int(0.15 * cfg.input_size[2])),
                        fill=0,
                        p=0.5),
        A.ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.7),
        A.Normalize(mean=cfg.mean, std=cfg.std),
        A.ToTensorV2(p=1.0)  # конвертируем numpy HxWxC в torch.Tensor CxHxW
    ]
)

df = pd.read_csv("data/items.csv")
ds = MultimodalDataset(df,
                       text_model=text_model,
                       image_model=image_model,
                       transforms=transforms)

loader = DataLoader(ds,
                    batch_size=1,
                    shuffle=False,
                    collate_fn=partial(collate_fn, tokenizer=tokenizer)) 

d:\Projects\praktikum\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import AutoModelForCausalLM
from transformers import AutoModel, AutoTokenizer

import torch
import torch.nn as nn

import timm


class BaseMultimodalModel(nn.Module):
    def __init__(self,
                 text_model_name='bert-base-uncased',
                 image_model_name='resnet50',
                 emb_dim=256):
        super().__init__()
        self.emb_dim = emb_dim
        self.text_model = AutoModel.from_pretrained(text_model_name)
        self.image_model = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0 
        )

        self.text_proj = nn.Linear(self.text_model.config.hidden_size, emb_dim)
        self.image_proj = nn.Linear(self.image_model.num_features, emb_dim)

    def forward(self, text_input, image_input):
        text_features = self.text_model(**text_input).last_hidden_state[:,  0, :]
        image_features = self.image_model(image_input)

        text_emb = self.text_proj(text_features)
        image_emb = self.image_proj(image_features)

        return text_emb, image_emb


text_model = 'bert-base-uncased'
image_model = 'resnet50'
m = BaseMultimodalModel(text_model_name=text_model,
                        image_model_name=image_model)


# 2 примера текста и картинки для инференса
tk = AutoTokenizer.from_pretrained(text_model)
tokenized = tk(["text", "text2"],
               return_tensors="pt",
               padding='max_length',
               truncation=True)

img = torch.randn(2, *m.image_model.pretrained_cfg["input_size"])
text_emb, image_emb = m(tokenized, img)
fused_emb = torch.cat([text_emb, image_emb], dim=1)
print(fused_emb.shape) 

d:\Projects\praktikum\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
